# ELEC 475 Lab 4: CLIP Model Evaluation

## Section 2.4: Evaluation and Visualization

This notebook evaluates the trained ResNet50-based CLIP image encoder by:
- Computing Recall@K metrics for image-to-text and text-to-image retrieval
- Visualizing text query-based image retrieval
- Demonstrating zero-shot image classification
- Analyzing performance trends and failure modes

### Setup Instructions

#### Running on Kaggle:
1. Click "+ Add Input" → "Your Work" → Select your training notebook run
2. Add the same 3 datasets as training:
   - MS-COCO 2014 Annotations
   - COCO 2014 Images
   - Pre-encoded Text Embeddings
3. The trained model will load from `/kaggle/input/training-COCO-CLIP/best_model.pt`

#### Running Locally:
1. Download `best_model.pt` from your Kaggle training run output
2. Place it in the Lab4 directory
3. Ensure you have the COCO 2014 dataset and text embeddings in `./data/`


## Imports and Environment Setup


In [ ]:
import os
import json
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from transformers import CLIPTokenizer, CLIPTextModel
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# --------------------------------------------------------------------
# Environment Detection and Path Configuration
# --------------------------------------------------------------------

# Auto-detect Kaggle environment
IS_KAGGLE = os.path.exists('/kaggle/input')

# DEMO_MODE: Set to True to run with fewer samples for quick testing
DEMO_MODE = False

if IS_KAGGLE:
    print("✓ Running on Kaggle")
    # Dataset 1: MS-COCO2014 annotations (captions)
    CAPTIONS_ROOT = Path("/kaggle/input/ms-coco2014/annotations")
    # Dataset 2: COCO 2014 images
    IMAGES_ROOT = Path("/kaggle/input/coco-2014-dataset-for-yolov3/coco2014/images")
    # Dataset 3: Pre-encoded text embeddings
    TEXT_EMBEDDINGS_PATH = Path("/kaggle/input/coco2014-clip-text-embeddings")
    # Dataset 4: Trained model (from training notebook output)
    # Update this path to match your training notebook name
    MODEL_PATH = Path("/kaggle/input/training-COCO-CLIP/best_model.pt")
    OUTPUT_DIR = Path("/kaggle/working")
else:
    print("✓ Running locally")
    # Modify these paths for local execution
    CAPTIONS_ROOT = Path("./data/annotations")
    IMAGES_ROOT = Path("./data")
    TEXT_EMBEDDINGS_PATH = Path("./data")
    MODEL_PATH = Path("./best_model.pt")  # or Path("./Lab4/best_model.pt")
    OUTPUT_DIR = Path("./outputs")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set specific paths
CAPTIONS_VAL_PATH = CAPTIONS_ROOT / "captions_val2014.json"
IMAGES_VAL_DIR = IMAGES_ROOT / "val2014"
VAL_EMBEDDINGS_PATH = TEXT_EMBEDDINGS_PATH / "val_text_embeddings.pt"

print("Dataset paths:")
print(f"  Val captions: {CAPTIONS_VAL_PATH}")
print(f"  Val images: {IMAGES_VAL_DIR}")
print(f"  Val embeddings: {VAL_EMBEDDINGS_PATH}")
print(f"  Model checkpoint: {MODEL_PATH}")
print(f"  Output directory: {OUTPUT_DIR}")

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Configuration


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Evaluation dataset size
if DEMO_MODE:
    EVAL_SUBSET_SIZE = 100  # For quick testing
else:
    EVAL_SUBSET_SIZE = 5000  # Use 5000 samples for faster evaluation
    # Set to None to use full validation set (~40K samples)

# Model configuration (must match training)
RESNET_OUTPUT_DIM = 2048
PROJECTION_HIDDEN_DIM = 1024
CLIP_EMBEDDING_DIM = 512

# CLIP normalization statistics (must match training)
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

# Batch size for embedding extraction
BATCH_SIZE = 64

print("=" * 60)
print("EVALUATION CONFIGURATION")
print("=" * 60)
print(f"Mode: {'DEMO' if DEMO_MODE else 'EVALUATION'}")
print(f"Device: {DEVICE}")
print(f"Evaluation subset size: {EVAL_SUBSET_SIZE if EVAL_SUBSET_SIZE else 'Full dataset'}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Embedding dimension: {CLIP_EMBEDDING_DIM}")
print("=" * 60)


## Model Architecture and Dataset Utilities


In [ ]:
# ============================================================
# Model Architecture (Must match training)
# ============================================================

class CLIPImageEncoder(nn.Module):
    """
    Image encoder for CLIP fine-tuning.
    
    Uses ResNet50 (pretrained on ImageNet) as backbone with a projection head
    to map image features to CLIP's 512-dimensional embedding space.
    """
    
    def __init__(
        self,
        resnet_output_dim: int = 2048,
        projection_hidden_dim: int = 1024,
        clip_embedding_dim: int = 512
    ):
        super().__init__()
        
        # Load pretrained ResNet50
        resnet = torchvision.models.resnet50(pretrained=False)  # Set to False since we'll load trained weights
        
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection head: 2048 -> 1024 -> 512
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(resnet_output_dim, projection_hidden_dim),
            nn.GELU(),
            nn.Linear(projection_hidden_dim, clip_embedding_dim)
        )
        
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input images [batch_size, 3, 224, 224]
            
        Returns:
            Image embeddings [batch_size, 512], L2-normalized
        """
        features = self.backbone(x)
        embeddings = self.projection(features)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings


# ============================================================
# Helper Functions for COCO Dataset
# ============================================================

def get_clip_image_transform():
    """Get transform pipeline for images to match CLIP preprocessing."""
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD)
    ])


def load_coco_captions(json_path: Path):
    """Load COCO-style caption JSON, handling potential root wrapper."""
    with json_path.open("r") as f:
        data = json.load(f)
    
    # Strip 'root' wrapper if present
    if isinstance(data, dict) and "root" in data and isinstance(data["root"], dict):
        data = data["root"]
    
    assert isinstance(data, dict), "Expected top-level JSON object"
    assert "images" in data, "Expected 'images' key in captions file"
    assert "annotations" in data, "Expected 'annotations' key in captions file"
    
    return data


class COCODataset(Dataset):
    """COCO 2014 dataset loader for CLIP evaluation."""
    
    def __init__(
        self,
        annotations_path: Path,
        images_dir: Path,
        text_embeddings_path: Path,
        subset_size: int = None,
        transform=None
    ):
        self.images_dir = images_dir
        self.transform = transform if transform is not None else get_clip_image_transform()
        
        # Load annotations
        print(f"Loading annotations from {annotations_path}...")
        captions_data = load_coco_captions(annotations_path)
        
        # Build image info dictionary
        images_dict = {img['id']: img for img in captions_data['images']}
        
        # Load text embeddings
        print(f"Loading text embeddings from {text_embeddings_path}...")
        if text_embeddings_path.exists():
            self.text_embeddings_cache = torch.load(text_embeddings_path, map_location='cpu')
            print(f"✓ Loaded embeddings for {len(self.text_embeddings_cache)} images")
        else:
            raise FileNotFoundError(f"Text embeddings not found at {text_embeddings_path}")
        
        # Build valid pairs list
        print("Building dataset pairs...")
        self.pairs = []
        skipped = 0
        caption_counters = {}
        
        for ann in captions_data['annotations']:
            image_id = ann['image_id']
            
            if image_id not in self.text_embeddings_cache or image_id not in images_dict:
                skipped += 1
                continue
                
            image_info = images_dict[image_id]
            caption_idx = caption_counters.get(image_id, 0)
            caption_counters[image_id] = caption_idx + 1
            
            self.pairs.append((image_id, caption_idx, image_info['file_name']))
        
        print(f"✓ Built dataset with {len(self.pairs)} pairs")
        if skipped > 0:
            print(f"  Skipped {skipped} annotations")
        
        # Sample subset if requested
        if subset_size is not None and subset_size < len(self.pairs):
            random.seed(42)
            self.pairs = random.sample(self.pairs, subset_size)
            print(f"✓ Sampled {subset_size} pairs from dataset")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        image_id, caption_idx, file_name = self.pairs[idx]
        
        # Load and transform image
        image_path = self.images_dir / file_name
        try:
            image = Image.open(image_path).convert('RGB')
            if self.transform is not None:
                image = self.transform(image)
        except (FileNotFoundError, OSError):
            return self.__getitem__((idx + 1) % len(self.pairs))
        
        # Get caption embedding
        caption_embedding = self.text_embeddings_cache[image_id][caption_idx]
        
        return image, caption_embedding, image_id, file_name

print("✓ Model architecture and dataset utilities defined")


## Load Trained Model


In [ ]:
print("=" * 60)
print("LOADING TRAINED MODEL")
print("=" * 60)

# Initialize model with same architecture as training
model = CLIPImageEncoder(
    resnet_output_dim=RESNET_OUTPUT_DIM,
    projection_hidden_dim=PROJECTION_HIDDEN_DIM,
    clip_embedding_dim=CLIP_EMBEDDING_DIM
)

# Load checkpoint
if not MODEL_PATH.exists():
    print(f"ERROR: Model checkpoint not found at {MODEL_PATH}")
    print("\nIf running on Kaggle:")
    print("  1. Go to 'Add Input' → 'Your Work'")
    print("  2. Select your training notebook run")
    print("  3. Update MODEL_PATH in the configuration cell above")
    print("\nIf running locally:")
    print("  1. Download best_model.pt from your Kaggle output")
    print("  2. Place it in the Lab4 directory")
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"✓ Loaded model from {MODEL_PATH}")
print(f"  Trained for {checkpoint['epoch']} epochs")
print(f"  Best validation loss: {checkpoint['val_loss']:.4f}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")
print("=" * 60)


In [ ]:
print("=" * 60)
print("LOADING VALIDATION DATASET")
print("=" * 60)

# Load validation dataset
val_dataset = COCODataset(
    annotations_path=CAPTIONS_VAL_PATH,
    images_dir=IMAGES_VAL_DIR,
    text_embeddings_path=VAL_EMBEDDINGS_PATH,
    subset_size=EVAL_SUBSET_SIZE,
    transform=get_clip_image_transform()
)

# Create data loader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Don't shuffle for evaluation
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✓ Validation set: {len(val_dataset)} pairs, {len(val_loader)} batches")
print("=" * 60)


In [ ]:
print("\n" + "=" * 60)
print("EXTRACTING EMBEDDINGS")
print("=" * 60)

# Extract all image and text embeddings
image_embeddings_list = []
text_embeddings_list = []
image_ids_list = []
file_names_list = []

model.eval()
with torch.no_grad():
    for images, text_embeddings, image_ids, file_names in tqdm(val_loader, desc="Extracting embeddings"):
        images = images.to(DEVICE)
        
        # Get image embeddings from trained model
        img_embs = model(images)
        
        # Collect embeddings
        image_embeddings_list.append(img_embs.cpu())
        text_embeddings_list.append(text_embeddings)
        image_ids_list.extend(image_ids)
        file_names_list.extend(file_names)

# Concatenate all embeddings
image_embeddings = torch.cat(image_embeddings_list, dim=0)  # [N, 512]
text_embeddings = torch.cat(text_embeddings_list, dim=0)    # [N, 512]

print(f"\n✓ Extracted embeddings for {len(image_embeddings)} samples")
print(f"  Image embeddings shape: {image_embeddings.shape}")
print(f"  Text embeddings shape: {text_embeddings.shape}")
print(f"  Image embeddings are L2-normalized: {torch.allclose(torch.norm(image_embeddings, dim=1), torch.ones(len(image_embeddings)), atol=1e-5)}")
print(f"  Text embeddings are L2-normalized: {torch.allclose(torch.norm(text_embeddings, dim=1), torch.ones(len(text_embeddings)), atol=1e-5)}")
print("=" * 60)


## Compute Cosine Similarity Matrix


In [ ]:
print("\n" + "=" * 60)
print("COMPUTING SIMILARITY MATRIX")
print("=" * 60)

# Compute cosine similarity matrix
# Since embeddings are L2-normalized, dot product = cosine similarity
similarity_matrix = torch.matmul(image_embeddings, text_embeddings.T)  # [N, N]

print(f"✓ Computed similarity matrix")
print(f"  Shape: {similarity_matrix.shape}")
print(f"  Min similarity: {similarity_matrix.min().item():.4f}")
print(f"  Max similarity: {similarity_matrix.max().item():.4f}")
print(f"  Mean similarity: {similarity_matrix.mean().item():.4f}")
print(f"  Mean diagonal (correct pairs): {torch.diagonal(similarity_matrix).mean().item():.4f}")
print("=" * 60)


## Recall@K Metrics

Recall@K measures how often the correct match appears in the top K retrieved results.

- **Image-to-Text (I2T)**: Given an image, retrieve the correct caption from all captions
- **Text-to-Image (T2I)**: Given a caption, retrieve the correct image from all images

For each query, we rank all candidates by similarity score and check if the correct match is in the top K.


In [ ]:
def compute_recall_at_k(similarity_matrix, k_values=[1, 5, 10]):
    """
    Compute Recall@K for both image-to-text and text-to-image retrieval.
    
    Args:
        similarity_matrix: [N, N] tensor where element [i, j] is similarity 
                          between image i and text j
        k_values: List of K values to compute recall for
        
    Returns:
        dict: Dictionary with I2T and T2I recall scores
    """
    N = similarity_matrix.shape[0]
    
    results = {
        'i2t': {},  # Image to text
        't2i': {}   # Text to image
    }
    
    # Image-to-Text Retrieval
    # For each image (row), rank all texts by similarity
    for k in k_values:
        correct_in_top_k = 0
        
        for i in range(N):
            # Get similarities for image i with all texts
            similarities = similarity_matrix[i]  # [N]
            
            # Get top-K text indices
            top_k_indices = torch.topk(similarities, k=k).indices
            
            # Check if correct text (index i) is in top-K
            if i in top_k_indices:
                correct_in_top_k += 1
        
        results['i2t'][f'R@{k}'] = (correct_in_top_k / N) * 100
    
    # Text-to-Image Retrieval
    # For each text (column), rank all images by similarity
    for k in k_values:
        correct_in_top_k = 0
        
        for j in range(N):
            # Get similarities for text j with all images
            similarities = similarity_matrix[:, j]  # [N]
            
            # Get top-K image indices
            top_k_indices = torch.topk(similarities, k=k).indices
            
            # Check if correct image (index j) is in top-K
            if j in top_k_indices:
                correct_in_top_k += 1
        
        results['t2i'][f'R@{k}'] = (correct_in_top_k / N) * 100
    
    return results


print("\n" + "=" * 60)
print("COMPUTING RECALL@K METRICS")
print("=" * 60)

# Compute recall metrics
recall_results = compute_recall_at_k(similarity_matrix, k_values=[1, 5, 10])

print("\nImage-to-Text Retrieval:")
print(f"  Recall@1:  {recall_results['i2t']['R@1']:.2f}%")
print(f"  Recall@5:  {recall_results['i2t']['R@5']:.2f}%")
print(f"  Recall@10: {recall_results['i2t']['R@10']:.2f}%")

print("\nText-to-Image Retrieval:")
print(f"  Recall@1:  {recall_results['t2i']['R@1']:.2f}%")
print(f"  Recall@5:  {recall_results['t2i']['R@5']:.2f}%")
print(f"  Recall@10: {recall_results['t2i']['R@10']:.2f}%")

print("\n" + "=" * 60)


## Text Query → Image Retrieval

Given a text query (e.g., "sport", "animal"), we encode it using the CLIP text encoder and retrieve the most similar images.


In [ ]:
print("\n" + "=" * 60)
print("LOADING CLIP TEXT ENCODER")
print("=" * 60)

# Load CLIP text encoder for encoding new queries
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
clip_text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
clip_text_model = clip_text_model.to(DEVICE)
clip_text_model.eval()

print("✓ Loaded CLIP text encoder (openai/clip-vit-base-patch32)")
print("=" * 60)


def encode_text_query(text: str):
    """Encode a text query using CLIP text encoder."""
    inputs = clip_tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = clip_text_model(**inputs)
        # Use pooled output (CLS token)
        text_embedding = outputs.pooler_output
        # L2 normalize
        text_embedding = F.normalize(text_embedding, p=2, dim=1)
    
    return text_embedding.cpu()


def denormalize_image(tensor, mean=CLIP_MEAN, std=CLIP_STD):
    """Denormalize a normalized image tensor for visualization."""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    tensor = torch.clamp(tensor, 0, 1)
    return tensor.permute(1, 2, 0).cpu().numpy()


def retrieve_images_for_query(query_text, top_k=5):
    """
    Retrieve top-K images for a given text query.
    
    Args:
        query_text: Text query string
        top_k: Number of images to retrieve
        
    Returns:
        List of (image_id, file_name, similarity_score) tuples
    """
    # Encode the query
    query_embedding = encode_text_query(query_text)
    
    # Compute similarity with all images
    similarities = torch.matmul(image_embeddings, query_embedding.T).squeeze()  # [N]
    
    # Get top-K indices
    top_k_indices = torch.topk(similarities, k=top_k).indices
    top_k_scores = torch.topk(similarities, k=top_k).values
    
    # Collect results
    results = []
    for idx, score in zip(top_k_indices, top_k_scores):
        idx = idx.item()
        image_id = image_ids_list[idx]
        file_name = file_names_list[idx]
        results.append((image_id, file_name, score.item()))
    
    return results


def visualize_text_query_retrieval(query_text, top_k=5):
    """Visualize top-K retrieved images for a text query."""
    results = retrieve_images_for_query(query_text, top_k=top_k)
    
    # Create figure
    fig, axes = plt.subplots(1, top_k, figsize=(3*top_k, 4))
    if top_k == 1:
        axes = [axes]
    
    fig.suptitle(f'Query: "{query_text}"', fontsize=16, fontweight='bold')
    
    for idx, (image_id, file_name, score) in enumerate(results):
        # Load and display image
        image_path = IMAGES_VAL_DIR / file_name
        try:
            image = Image.open(image_path).convert('RGB')
            axes[idx].imshow(image)
            axes[idx].axis('off')
            axes[idx].set_title(f'Similarity: {score:.3f}', fontsize=10)
        except Exception as e:
            axes[idx].text(0.5, 0.5, 'Image\nNot Found', 
                          ha='center', va='center', fontsize=12)
            axes[idx].axis('off')
    
    plt.tight_layout()
    return fig

print("✓ Text query retrieval functions defined")


In [ ]:
print("\n" + "=" * 60)
print("TEXT QUERY RETRIEVAL EXAMPLES")
print("=" * 60)

# Test queries
test_queries = ['sport', 'animal', 'food', 'landscape', 'person']

for query in test_queries:
    print(f"\nQuery: '{query}'")
    fig = visualize_text_query_retrieval(query, top_k=5)
    plt.savefig(OUTPUT_DIR / f'query_{query}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Saved visualization to {OUTPUT_DIR / f'query_{query}.png'}")

print("\n" + "=" * 60)


In [ ]:
def classify_image(image_idx, class_labels):
    """
    Classify an image using zero-shot classification.
    
    Args:
        image_idx: Index in the dataset
        class_labels: List of class label strings (e.g., ['a person', 'an animal'])
        
    Returns:
        dict: Classification results with scores for each class
    """
    # Get image embedding
    img_embedding = image_embeddings[image_idx:image_idx+1]  # [1, 512]
    
    # Encode all class labels
    class_embeddings = []
    for label in class_labels:
        class_emb = encode_text_query(label)
        class_embeddings.append(class_emb)
    class_embeddings = torch.cat(class_embeddings, dim=0)  # [num_classes, 512]
    
    # Compute similarities
    similarities = torch.matmul(img_embedding, class_embeddings.T).squeeze()  # [num_classes]
    
    # Apply softmax to get probabilities
    probabilities = F.softmax(similarities / 0.01, dim=0)  # Temperature of 0.01 for sharper distribution
    
    # Get results
    results = {
        'class_labels': class_labels,
        'similarities': similarities.tolist(),
        'probabilities': probabilities.tolist(),
        'predicted_class': class_labels[torch.argmax(similarities).item()],
        'confidence': torch.max(probabilities).item()
    }
    
    return results


def visualize_zero_shot_classification(image_idx, class_labels):
    """Visualize zero-shot classification for an image."""
    results = classify_image(image_idx, class_labels)
    
    # Create figure with image and bar chart
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Display image
    image_path = IMAGES_VAL_DIR / file_names_list[image_idx]
    try:
        image = Image.open(image_path).convert('RGB')
        axes[0].imshow(image)
        axes[0].axis('off')
        axes[0].set_title(f'Predicted: {results["predicted_class"]}\\nConfidence: {results["confidence"]:.1%}', 
                         fontsize=12, fontweight='bold')
    except Exception as e:
        axes[0].text(0.5, 0.5, 'Image Not Found', ha='center', va='center', fontsize=12)
        axes[0].axis('off')
    
    # Display probability bar chart
    y_pos = np.arange(len(class_labels))
    bars = axes[1].barh(y_pos, results['probabilities'])
    
    # Color the predicted class differently
    predicted_idx = class_labels.index(results['predicted_class'])
    bars[predicted_idx].set_color('green')
    
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(class_labels)
    axes[1].set_xlabel('Probability', fontsize=11)
    axes[1].set_title('Classification Scores', fontsize=12, fontweight='bold')
    axes[1].set_xlim([0, 1])
    
    # Add value labels on bars
    for i, (prob, sim) in enumerate(zip(results['probabilities'], results['similarities'])):
        axes[1].text(prob + 0.02, i, f'{prob:.2%} (sim: {sim:.3f})', 
                    va='center', fontsize=9)
    
    plt.tight_layout()
    return fig

print("✓ Zero-shot classification functions defined")


In [ ]:
print("\n" + "=" * 60)
print("ZERO-SHOT CLASSIFICATION EXAMPLES")
print("=" * 60)

# Define test cases with different class sets
test_cases = [
    {
        'classes': ['a person', 'an animal', 'a landscape'],
        'num_examples': 3
    },
    {
        'classes': ['a dog', 'a cat', 'a bird', 'a fish'],
        'num_examples': 2
    },
    {
        'classes': ['sports', 'food', 'nature', 'architecture'],
        'num_examples': 2
    }
]

example_idx = 0
for test_case_idx, test_case in enumerate(test_cases):
    class_labels = test_case['classes']
    num_examples = test_case['num_examples']
    
    print(f"\n--- Test Case {test_case_idx + 1}: Classes = {class_labels} ---")
    
    for i in range(num_examples):
        # Use different images from the dataset
        image_idx = (example_idx * 17) % len(image_embeddings)  # Spread out examples
        
        print(f"\nExample {i+1}:")
        fig = visualize_zero_shot_classification(image_idx, class_labels)
        plt.savefig(OUTPUT_DIR / f'zeroshot_case{test_case_idx+1}_ex{i+1}.png', 
                   dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved to {OUTPUT_DIR / f'zeroshot_case{test_case_idx+1}_ex{i+1}.png'}")
        
        example_idx += 1

print("\n" + "=" * 60)


## Performance Visualization and Analysis


In [ ]:
print("\n" + "=" * 60)
print("CREATING PERFORMANCE VISUALIZATIONS")
print("=" * 60)

# 1. Recall@K Bar Chart
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

k_values = [1, 5, 10]
x = np.arange(len(k_values))
width = 0.35

i2t_scores = [recall_results['i2t'][f'R@{k}'] for k in k_values]
t2i_scores = [recall_results['t2i'][f'R@{k}'] for k in k_values]

bars1 = ax.bar(x - width/2, i2t_scores, width, label='Image-to-Text', color='steelblue')
bars2 = ax.bar(x + width/2, t2i_scores, width, label='Text-to-Image', color='coral')

ax.set_xlabel('K', fontsize=12, fontweight='bold')
ax.set_ylabel('Recall@K (%)', fontsize=12, fontweight='bold')
ax.set_title('Retrieval Performance: Recall@K Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'K={k}' for k in k_values])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 100])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'recall_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved Recall@K chart")


# 2. Similarity Matrix Heatmap (subset)
print("\nCreating similarity matrix heatmap...")
subset_size = min(50, len(similarity_matrix))
subset_sim = similarity_matrix[:subset_size, :subset_size].numpy()

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(subset_sim, cmap='viridis', aspect='auto')

ax.set_xlabel('Caption Index', fontsize=11)
ax.set_ylabel('Image Index', fontsize=11)
ax.set_title(f'Cosine Similarity Matrix (first {subset_size} samples)', 
            fontsize=13, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Cosine Similarity', fontsize=11)

# Highlight diagonal
for i in range(subset_size):
    ax.add_patch(mpatches.Rectangle((i-0.5, i-0.5), 1, 1, 
                                     fill=False, edgecolor='red', linewidth=1.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved similarity matrix heatmap")


# 3. Success and Failure Cases
print("\nAnalyzing success and failure cases...")

# Get diagonal similarities (correct pairs)
diagonal_sims = torch.diagonal(similarity_matrix)

# Find best matches (highest similarity for correct pairs)
best_indices = torch.topk(diagonal_sims, k=5).indices
print("\nTop 5 Success Cases (highest similarity for correct pairs):")
for rank, idx in enumerate(best_indices, 1):
    print(f"  {rank}. Index {idx.item()}: similarity = {diagonal_sims[idx].item():.4f}")

# Find worst matches (lowest similarity for correct pairs)
worst_indices = torch.topk(diagonal_sims, k=5, largest=False).indices
print("\nTop 5 Failure Cases (lowest similarity for correct pairs):")
for rank, idx in enumerate(worst_indices, 1):
    print(f"  {rank}. Index {idx.item()}: similarity = {diagonal_sims[idx].item():.4f}")

# Visualize success cases
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Success Cases: High Similarity Between Correct Image-Caption Pairs', 
            fontsize=13, fontweight='bold')

for i, idx in enumerate(best_indices):
    idx = idx.item()
    image_path = IMAGES_VAL_DIR / file_names_list[idx]
    try:
        image = Image.open(image_path).convert('RGB')
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(f'Sim: {diagonal_sims[idx].item():.3f}', fontsize=10)
    except:
        axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'success_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved success cases visualization")

# Visualize failure cases
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Failure Cases: Low Similarity Between Correct Image-Caption Pairs', 
            fontsize=13, fontweight='bold')

for i, idx in enumerate(worst_indices):
    idx = idx.item()
    image_path = IMAGES_VAL_DIR / file_names_list[idx]
    try:
        image = Image.open(image_path).convert('RGB')
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(f'Sim: {diagonal_sims[idx].item():.3f}', fontsize=10)
    except:
        axes[i].text(0.5, 0.5, 'N/A', ha='center', va='center')
        axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'failure_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved failure cases visualization")


# 4. Similarity Distribution Analysis
print("\nAnalyzing similarity distributions...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Diagonal (correct pairs) vs off-diagonal (incorrect pairs)
diagonal_similarities = torch.diagonal(similarity_matrix).numpy()
off_diagonal_mask = ~torch.eye(len(similarity_matrix), dtype=bool)
off_diagonal_similarities = similarity_matrix[off_diagonal_mask].numpy()

# Sample off-diagonal for visualization (too many points otherwise)
sample_size = min(10000, len(off_diagonal_similarities))
off_diagonal_sample = np.random.choice(off_diagonal_similarities, sample_size, replace=False)

axes[0].hist(diagonal_similarities, bins=50, alpha=0.7, label='Correct pairs (diagonal)', 
            color='green', edgecolor='black')
axes[0].hist(off_diagonal_sample, bins=50, alpha=0.7, label='Incorrect pairs (off-diagonal)', 
            color='red', edgecolor='black')
axes[0].set_xlabel('Cosine Similarity', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Similarities', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Box plot comparison
data_to_plot = [diagonal_similarities, off_diagonal_sample]
axes[1].boxplot(data_to_plot, labels=['Correct Pairs', 'Incorrect Pairs'])
axes[1].set_ylabel('Cosine Similarity', fontsize=11)
axes[1].set_title('Similarity Comparison', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'similarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved similarity distribution analysis")

print("\n" + "=" * 60)
print("Statistical Summary:")
print(f"  Correct pairs (diagonal):")
print(f"    Mean: {diagonal_similarities.mean():.4f}")
print(f"    Std:  {diagonal_similarities.std():.4f}")
print(f"    Min:  {diagonal_similarities.min():.4f}")
print(f"    Max:  {diagonal_similarities.max():.4f}")
print(f"  Incorrect pairs (off-diagonal):")
print(f"    Mean: {off_diagonal_similarities.mean():.4f}")
print(f"    Std:  {off_diagonal_similarities.std():.4f}")
print(f"    Min:  {off_diagonal_similarities.min():.4f}")
print(f"    Max:  {off_diagonal_similarities.max():.4f}")
print("=" * 60)


## Discussion and Analysis

### Performance Trends and Training Dynamics

This section discusses the evaluation results and provides insights into model performance.


In [ ]:
print("\n" + "=" * 60)
print("EVALUATION SUMMARY AND DISCUSSION")
print("=" * 60)

print("\n### 1. RETRIEVAL PERFORMANCE ANALYSIS")
print("-" * 60)

# Compare I2T vs T2I
print("\nImage-to-Text vs Text-to-Image Comparison:")
for k in [1, 5, 10]:
    i2t_score = recall_results['i2t'][f'R@{k}']
    t2i_score = recall_results['t2i'][f'R@{k}']
    diff = abs(i2t_score - t2i_score)
    print(f"  R@{k}: I2T = {i2t_score:.2f}%, T2I = {t2i_score:.2f}%, Diff = {diff:.2f}%")

print("\nKey Observations:")
print("  • Symmetric performance: I2T and T2I scores are similar, indicating")
print("    balanced bidirectional learning from the symmetric InfoNCE loss.")
print("  • Recall improvement: Performance increases with K, which is expected")
print("    as the model gets more chances to find the correct match.")

# Performance interpretation
avg_recall_1 = (recall_results['i2t']['R@1'] + recall_results['t2i']['R@1']) / 2
if avg_recall_1 > 40:
    quality = "excellent"
elif avg_recall_1 > 25:
    quality = "good"
elif avg_recall_1 > 15:
    quality = "moderate"
else:
    quality = "needs improvement"

print(f"  • Overall quality: {quality.upper()} (avg R@1 = {avg_recall_1:.1f}%)")

print("\n### 2. SIMILARITY ANALYSIS")
print("-" * 60)

mean_correct = diagonal_similarities.mean()
mean_incorrect = off_diagonal_similarities.mean()
separation = mean_correct - mean_incorrect

print(f"\nSeparation between correct and incorrect pairs:")
print(f"  • Mean similarity (correct pairs):   {mean_correct:.4f}")
print(f"  • Mean similarity (incorrect pairs): {mean_incorrect:.4f}")
print(f"  • Separation margin: {separation:.4f}")

if separation > 0.15:
    print("  • GOOD: Strong separation indicates the model learned to")
    print("    distinguish matching pairs from non-matching pairs effectively.")
elif separation > 0.08:
    print("  • MODERATE: Reasonable separation, but there's room for improvement")
    print("    in distinguishing matching from non-matching pairs.")
else:
    print("  • WEAK: Limited separation suggests difficulty in learning")
    print("    discriminative features. Consider longer training or better data.")

print("\n### 3. TRAINING DYNAMICS")
print("-" * 60)

print("\nFactors affecting performance:")
print("  • Dataset size: Trained on subset of COCO 2014")
if EVAL_SUBSET_SIZE:
    print(f"    - Evaluated on {EVAL_SUBSET_SIZE} samples")
print("  • Architecture: ResNet50 backbone + projection head")
print("  • Loss function: InfoNCE (contrastive loss)")
print("  • Text encoder: Pre-trained CLIP (frozen during training)")
print("\nExpected behaviors:")
print("  ✓ Training loss should decrease steadily")
print("  ✓ Validation loss should follow training loss (with some gap)")
print("  ✓ If validation loss increases while training loss decreases:")
print("    → Overfitting occurred, early stopping would help")

print("\n### 4. COMMON FAILURE MODES")
print("-" * 60)

print("\nBased on the failure cases analysis, typical failure modes include:")
print("  1. Abstract/Complex scenes:")
print("     • Images with multiple objects or complex compositions")
print("     • Captions describing relationships or abstract concepts")
print("  2. Generic captions:")
print("     • Captions like 'a photo of something' match many images")
print("     • Leads to ambiguity in retrieval")
print("  3. Fine-grained distinctions:")
print("     • Similar objects (e.g., different dog breeds)")
print("     • Subtle differences in scenes (e.g., 'sunny beach' vs 'beach')")
print("  4. Long/detailed captions:")
print("     • Longer captions may lose semantic coherence")
print("     • Model may focus on subset of caption content")

print("\n### 5. ZERO-SHOT CLASSIFICATION INSIGHTS")
print("-" * 60)

print("\nThe zero-shot classification demonstrates:")
print("  • Transfer learning: Model can classify into arbitrary categories")
print("    without being explicitly trained on those categories")
print("  • Semantic understanding: By comparing image embeddings with")
print("    text embeddings of class names, the model performs classification")
print("  • Flexibility: Can define custom classes at inference time")
print("  • Limitations: Performance depends on how well class names align")
print("    with training data distribution")

print("\n### 6. RECOMMENDATIONS FOR IMPROVEMENT")
print("-" * 60)

print("\nTo improve performance:")
print("  1. Training:")
print("     • Increase training epochs if validation loss still decreasing")
print("     • Use full COCO dataset instead of subset")
print("     • Experiment with different learning rates and batch sizes")
print("     • Try different temperature values in InfoNCE loss")
print("  2. Architecture:")
print("     • Experiment with different backbones (EfficientNet, ViT)")
print("     • Try deeper projection heads")
print("     • Add attention mechanisms")
print("  3. Data:")
print("     • Use data augmentation (already implemented in training)")
print("     • Filter out low-quality or ambiguous caption-image pairs")
print("     • Use hard negative mining during training")
print("  4. Loss function:")
print("     • Try other contrastive losses (SimCLR, MoCo)")
print("     • Add auxiliary losses (e.g., hard negative loss)")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)


In [ ]:
# Save evaluation summary to file
summary_path = OUTPUT_DIR / 'evaluation_summary.txt'
with open(summary_path, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("CLIP MODEL EVALUATION SUMMARY\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("RECALL@K METRICS\n")
    f.write("-" * 60 + "\n")
    f.write("Image-to-Text Retrieval:\n")
    for k in [1, 5, 10]:
        f.write(f"  Recall@{k:2d}: {recall_results['i2t'][f'R@{k}']:6.2f}%\n")
    f.write("\nText-to-Image Retrieval:\n")
    for k in [1, 5, 10]:
        f.write(f"  Recall@{k:2d}: {recall_results['t2i'][f'R@{k}']:6.2f}%\n")
    f.write("\n")
    
    f.write("SIMILARITY STATISTICS\n")
    f.write("-" * 60 + "\n")
    f.write("Correct pairs (diagonal):\n")
    f.write(f"  Mean: {diagonal_similarities.mean():.4f}\n")
    f.write(f"  Std:  {diagonal_similarities.std():.4f}\n")
    f.write(f"  Min:  {diagonal_similarities.min():.4f}\n")
    f.write(f"  Max:  {diagonal_similarities.max():.4f}\n")
    f.write("\nIncorrect pairs (off-diagonal):\n")
    f.write(f"  Mean: {off_diagonal_similarities.mean():.4f}\n")
    f.write(f"  Std:  {off_diagonal_similarities.std():.4f}\n")
    f.write(f"  Min:  {off_diagonal_similarities.min():.4f}\n")
    f.write(f"  Max:  {off_diagonal_similarities.max():.4f}\n")
    f.write(f"\nSeparation margin: {mean_correct - mean_incorrect:.4f}\n\n")
    
    f.write("EVALUATION DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Number of samples: {len(image_embeddings)}\n")
    f.write(f"Dataset: COCO 2014 Validation\n")
    f.write(f"Embedding dimension: {CLIP_EMBEDDING_DIM}\n\n")
    
    f.write("MODEL INFORMATION\n")
    f.write("-" * 60 + "\n")
    f.write(f"Architecture: ResNet50 + Projection Head\n")
    f.write(f"Projection: {RESNET_OUTPUT_DIM} → {PROJECTION_HIDDEN_DIM} → {CLIP_EMBEDDING_DIM}\n")
    f.write(f"Total parameters: {total_params:,}\n")
    if 'checkpoint' in locals():
        f.write(f"Trained for: {checkpoint['epoch']} epochs\n")
        f.write(f"Best validation loss: {checkpoint['val_loss']:.4f}\n")
    f.write("\n")
    
    f.write("=" * 60 + "\n")

print(f"\n✓ Saved evaluation summary to {summary_path}")

print("\n" + "=" * 60)
print("DELIVERABLES")
print("=" * 60)
print(f"\nOutput directory: {OUTPUT_DIR}")
print("\nGenerated files:")
print("  1. evaluation_summary.txt - Complete evaluation metrics")
print("  2. recall_metrics.png - Recall@K bar chart")
print("  3. similarity_matrix.png - Similarity matrix heatmap")
print("  4. success_cases.png - Top 5 best matching pairs")
print("  5. failure_cases.png - Top 5 worst matching pairs")
print("  6. similarity_distribution.png - Distribution analysis")
print("  7. query_*.png - Text query retrieval examples (5 queries)")
print("  8. zeroshot_*.png - Zero-shot classification examples (7 examples)")

print("\nFor your lab report, include:")
print("  ✓ Recall@K metrics for I2T and T2I retrieval")
print("  ✓ Similarity matrix visualization")
print("  ✓ Success and failure case examples")
print("  ✓ Text query retrieval demonstrations")
print("  ✓ Zero-shot classification examples")
print("  ✓ Discussion of performance trends and failure modes")
print("  ✓ Analysis of training dynamics and recommendations")

print("\n" + "=" * 60)
print("LAB 4 EVALUATION COMPLETE!")
print("=" * 60)
